# Weather Intelligence Platform - Next-Hour ML Weather Prediction

## 1. Problem Formulation
Predicting weather conditions for the **next hour ($T + 1$)** using historical telemetry observations up to time $T$.

- **Primary Target**: Next-hour continuous temperature ($^\circ\text{C}$)
- **Secondary Target**: Next-hour precipitation occurrence ($> 0.1\text{ mm}$)
- **Critical Constraints**:
  - Strict chronological splitting (no shuffling, no future data leakage)
  - Comparison against a persistence baseline ($y_{T+1} = y_T$)
  - Transparent evaluation metrics (MAE, RMSE, $R^2$, Accuracy, Precision, Recall, F1)

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add root project to sys.path
sys.path.insert(0, os.path.abspath('..'))

from utils.dataset import load_or_fetch_dataset, chronological_split
from features.feature_engineering import prepare_feature_matrix, FEATURE_COLUMNS
from utils.metrics import calculate_regression_metrics, calculate_classification_metrics

print("Libraries successfully loaded.")

## 2. Dataset Loading & Validation

In [ ]:
df = load_or_fetch_dataset(min_rows=500)
print(f"Dataset shape: {df.shape}")
print(f"Time span: {df['recorded_at'].min()} to {df['recorded_at'].max()}")
df.head()

## 3. Feature Engineering & Target Alignment
Extract time cyclical encodings, historical lag variables ($T-1, T-2, T-3, T-6$), and rolling statistical windows.

In [ ]:
X, y_temp, y_rain = prepare_feature_matrix(df)
print(f"Constructed feature matrix: {X.shape}")
print(f"Features list ({len(FEATURE_COLUMNS)}): {FEATURE_COLUMNS}")

## 4. Chronological Train / Validation / Test Splitting

In [ ]:
X_train, X_val, X_test, y_train_t, y_val_t, y_test_t = chronological_split(X, y_temp, 0.70, 0.15)
print(f"Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}")

## 5. Model Evaluation: Temperature Regressors
Compare Persistence Baseline, Linear Regression, and Random Forest Regressor.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# 1. Persistence Baseline
base_pred = X_test['temperature'].values
base_m = calculate_regression_metrics(y_test_t, base_pred)

# 2. Linear Regression
lr = LinearRegression().fit(X_train, y_train_t)
lr_pred = lr.predict(X_test)
lr_m = calculate_regression_metrics(y_test_t, lr_pred)

# 3. Random Forest Regressor
rf = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42).fit(X_train, y_train_t)
rf_pred = rf.predict(X_test)
rf_m = calculate_regression_metrics(y_test_t, rf_pred)

comparison = pd.DataFrame([
    {"Model": "Persistence Baseline", **base_m},
    {"Model": "Linear Regression", **lr_m},
    {"Model": "Random Forest Regressor", **rf_m},
])
comparison

## 6. Feature Importance
Analyze which atmospheric variables contribute most to next-hour predictions.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=False)
print("Top 10 Feature Importances:")
print(importances.head(10))